# 01 — Data Quality & Validation
## SOXL Risk Analysis Project

### Purpose
Before analysing SOXL's downside risk, we need to verify that our historical market data is accurate, complete, and consistent.

This notebook documents the data validation process for four datasets:

- **SOXL:** Direxion Daily Semiconductor Bull 3X Shares
- **SOXS:** Direxion Daily Semiconductor Bear 3X Shares
- **SOXX:** iShares Semiconductor ETF
- **NASDAQSOX:** PHLX Semiconductor Sector Index, retrieved from FRED

### Validation Checklist
1. Data completeness and missing values
2. Trading-date alignment
3. Stock splits and dividend adjustments
4. Extreme daily returns
5. Corporate action verification
6. Final data-quality decisions

**Historical period:** March 2010 – October 30, 2026

**Data sources:** Yahoo Finance and Federal Reserve Economic Data (FRED)

**Status:** Validation in progress


In [3]:
from pathlib import Path
import pandas as pd

# Jupyter is running from the project's root directory.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Load the original historical datasets.
soxl = pd.read_csv(DATA_DIR / "soxl_yahoo_daily.csv")
soxs = pd.read_csv(DATA_DIR / "soxs_yahoo_daily.csv")
soxx = pd.read_csv(DATA_DIR / "soxx_yahoo_daily.csv")
fred = pd.read_csv(DATA_DIR / "nasdaqsox_fred_original.csv")

# Display the number of observations in each dataset.
summary = pd.DataFrame({
    "Dataset": ["SOXL", "SOXS", "SOXX", "NASDAQSOX"],
    "Observations": [len(soxl), len(soxs), len(soxx), len(fred)]
})

display(summary)

,Dataset,Observations
0,SOXL,4187
1,SOXS,4187
2,SOXX,4195
3,NASDAQSOX,4350


## 1. Initial Data Quality Assessment

### Objective
Check all four datasets for missing values, duplicate dates,
invalid dates, and nonpositive prices.

### Why this matters
Errors in historical prices can distort return calculations,
volatility estimates, and downside risk measurements.

We will preserve the original downloaded data and investigate
any issues before making cleaning decisions.

In [4]:
# Organize our four datasets into a dictionary.
datasets = {
    "SOXL": soxl,
    "SOXS": soxs,
    "SOXX": soxx,
    "NASDAQSOX": fred
}

results = []

for name, df in datasets.items():

    # FRED uses a different date column name.
    date_col = "observation_date" if name == "NASDAQSOX" else "Date"

    dates = pd.to_datetime(df[date_col], errors="coerce")

    results.append({
        "Dataset": name,
        "Total Rows": len(df),
        "Missing Values": df.isna().sum().sum(),
        "Duplicate Dates": dates.duplicated().sum(),
        "Invalid Dates": dates.isna().sum()
    })

quality_summary = pd.DataFrame(results)

display(quality_summary)

,Dataset,Total Rows,Missing Values,Duplicate Dates,Invalid Dates
0,SOXL,4187,0,0,0
1,SOXS,4187,0,0,0
2,SOXX,4195,0,0,0
3,NASDAQSOX,4350,155,0,0


### 1.1 Investigating Missing FRED Observations

The initial assessment identified 155 missing NASDAQSOX values.

These may represent non-trading days, such as stock market holidays, rather than errors in the dataset.

To investigate, we compare the dates with valid FRED observations against SOXX's trading dates.

**Validation criteria:** If both datasets contain the same trading dates, we can exclude the missing FRED observations during data cleaning without inventing replacement prices.

In [5]:
# Convert both date columns into datetime format.
fred_dates = pd.to_datetime(fred["observation_date"])
soxx_dates = pd.to_datetime(soxx["Date"])

# Keep only FRED dates with an actual index value.
fred_valid_dates = set(
    fred_dates[fred["NASDAQSOX"].notna()]
)

# Collect SOXX trading dates.
soxx_trading_dates = set(soxx_dates)

# Compare the two trading calendars.
calendar_results = pd.DataFrame({
    "Check": [
        "FRED missing observations",
        "FRED valid trading dates",
        "SOXX trading dates",
        "FRED dates absent from SOXX",
        "SOXX dates absent from FRED"
    ],
    "Result": [
        fred["NASDAQSOX"].isna().sum(),
        len(fred_valid_dates),
        len(soxx_trading_dates),
        len(fred_valid_dates - soxx_trading_dates),
        len(soxx_trading_dates - fred_valid_dates)
    ]
})

display(calendar_results)

# Display examples of dates with missing FRED values.
missing_fred = fred[fred["NASDAQSOX"].isna()]

display(missing_fred.head(10))

,Check,Result
0,FRED missing observations,155
1,FRED valid trading dates,4195
2,SOXX trading dates,4195
3,FRED dates absent from SOXX,0
4,SOXX dates absent from FRED,0


,observation_date,NASDAQSOX
24,2010-04-02,NaN
65,2010-05-31,NaN
90,2010-07-05,NaN
135,2010-09-06,NaN
193,2010-11-25,NaN
214,2010-12-24,NaN
230,2011-01-17,NaN
255,2011-02-21,NaN
299,2011-04-22,NaN
325,2011-05-30,NaN


### Findings and Cleaning Decision

The FRED dataset contains 155 missing index values. Investigation shows that all 4,195 valid FRED observation dates match SOXX's trading calendar exactly.

**Conclusion:** The missing observations occur outside the trading calendar represented by SOXX. They do not indicate missing index prices on those trading days.

**Cleaning decision:**
- Preserve the original FRED dataset unchanged.
- Exclude rows with missing index values when preparing the cleaned dataset.
- Never replace missing prices with zero.

This ensures that our calculations use actual market observations without introducing artificial returns.

## 2. ETF Trading-Date Alignment

### Objective
Verify that SOXL, SOXS and SOXX contain observations for the same trading days.

### Why this matters
Comparing returns from different dates would produce misleading results.

We will identify the common trading period and determine whether any ETFs have additional observations outside that period.

In [7]:
# Convert ETF date columns into sets for comparison.
etf_dates = {}

for ticker, df in {
    "SOXL": soxl,
    "SOXS": soxs,
    "SOXX": soxx
}.items():
    etf_dates[ticker] = set(pd.to_datetime(df["Date"]))

# Find dates shared by all three ETFs.
common_dates = set.intersection(*etf_dates.values())

# Create a summary table.
alignment_summary = pd.DataFrame({
    "ETF": ["SOXL", "SOXS", "SOXX"],
    "Total Trading Days": [
        len(etf_dates["SOXL"]),
        len(etf_dates["SOXS"]),
        len(etf_dates["SOXX"])
    ],
    "Shared Trading Days": [len(common_dates)] * 3,
    "Additional Dates": [
        len(etf_dates[ticker] - common_dates)
        for ticker in ["SOXL", "SOXS", "SOXX"]
    ]
})

display(alignment_summary)

print("First shared date:", min(common_dates).date())
print("Last shared date:", max(common_dates).date())
print("SOXL and SOXS calendars match:",
      etf_dates["SOXL"] == etf_dates["SOXS"])

,ETF,Total Trading Days,Shared Trading Days,Additional Dates
0,SOXL,4187,4187,0
1,SOXS,4187,4187,0
2,SOXX,4195,4187,8


First shared date: 2010-03-11
Last shared date: 2026-10-30
SOXL and SOXS calendars match: True


### Findings and Methodology Decision

All three ETFs share 4,187 trading dates between March 11, 2010 and October 30, 2026.

SOXL and SOXS have identical trading calendars. SOXX contains eight additional observations before the other two ETFs begin.

**Decision:** Use the 4,187 common dates when comparing all three ETFs. The datasets will be joined by trading date rather than row position.

This preserves the original datasets while ensuring that daily observations are comparable.

## 3. Stock Split Validation

### Objective
Identify historical stock splits and verify that they do not create artificial investment gains or losses in our calculations.

### Why this matters
A stock split changes the number of shares and price per share without directly changing the total value of an investment.

For example, a 4-for-1 forward split converts one $100 share into four $25 shares.

A reverse split has the opposite effect.

If historical prices are not adjusted for these changes, calculated returns and volatility may be misleading.

We will first identify the recorded split events for SOXL, SOXS and SOXX.

In [8]:
from fractions import Fraction

split_results = []

for ticker, df in {
    "SOXL": soxl,
    "SOXS": soxs,
    "SOXX": soxx
}.items():

    # Select dates when a stock split occurred.
    splits = df[df["Stock Splits"] > 0]

    for _, event in splits.iterrows():

        ratio = event["Stock Splits"]

        # Convert the numerical ratio into a readable form.
        fraction = Fraction(float(ratio)).limit_denominator(100)

        split_results.append({
            "ETF": ticker,
            "Date": event["Date"],
            "Split Ratio": f"{fraction.numerator}-for-{fraction.denominator}",
            "Type": "Forward" if ratio > 1 else "Reverse"
        })

split_summary = pd.DataFrame(split_results)

display(split_summary)

print("\nTotal split events:", len(split_summary))

,ETF,Date,Split Ratio,Type
0,SOXL,2015-05-20,4-for-1,Forward
1,SOXL,2021-03-02,15-for-1,Forward
2,SOXS,2011-02-24,1-for-5,Reverse
3,SOXS,2013-08-20,1-for-4,Reverse
4,SOXS,2015-05-20,1-for-4,Reverse
5,SOXS,2017-05-01,1-for-5,Reverse
6,SOXS,2019-06-28,1-for-10,Reverse
7,SOXS,2020-08-28,1-for-12,Reverse
8,SOXS,2022-03-28,1-for-10,Reverse
9,SOXS,2024-04-15,1-for-10,Reverse



Total split events: 13
